# Resume ↔ Job Description Matching (src-backed)

This notebook is the **repository-native** version of `01_ranking_experiments_colab.ipynb`. It runs the *same experiment* but imports every piece of logic from the project's `src/` package instead of redefining it inline — so the notebook is pure orchestration and the algorithms are unit-tested. This is the version pushed to GitHub.

We benchmark three approaches against the ground-truth `matched_score` in the [Resume Dataset](https://www.kaggle.com/datasets/saugataroyarghya/resume-dataset/data).

| # | Method | Family | Supervised? |
|---|--------|--------|-------------|
| 1 | **TF-IDF + cosine similarity** — sparse lexical baseline. | Lexical | No |
| 2 | **SBERT bi-encoder + cosine** (`all-MiniLM-L6-v2`). | Dense | No |
| 3 | **Fine-tuned cross-encoder** (`ms-marco-MiniLM-L-6-v2`) trained on `matched_score` with MSE loss. | Pair model | **Yes** |

All three are evaluated on the **same held-out test split** using regression metrics (Pearson, Spearman, MAE, RMSE) and ranking metrics (NDCG@10, Precision@10, MRR).

**Runtime:** Section 9 fine-tunes the cross-encoder (~25–35 min on a GPU; slow on CPU). If a trained model already exists at `outputs/models/cross_encoder_resume_jd/`, training is skipped and the cached model is reused.

## 1. Install dependencies

Pinned in the repo's `requirements.txt`.

In [ ]:
%pip install -q -r ../requirements.txt

## 2. Imports and environment check

The project root is added to `sys.path` so `import src...` resolves when the notebook runs from `notebooks/`.

In [ ]:
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src import config
from src.data_loader import load_dataset, build_composite_text
from src.preprocessing import clean_text
from src.rankers.lexical import score_lexical
from src.rankers.semantic import score_semantic
from src.rankers.cross_encoder import (
    stratified_split,
    train_cross_encoder,
    score_cross_encoder,
)
from src.evaluation import regression_metrics, ranking_metrics
from src.explainability import top_shared_terms

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {_PROJECT_ROOT}")
print(f"Device:       {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
else:
    print("WARNING:      No GPU detected. Section 9 training will be slow on CPU.")
print(f"PyTorch:      {torch.__version__}")

## 3. Configuration

All knobs live in `src/config.py` — a single source of truth shared by the notebook, the unit tests, and the Streamlit app.

In [ ]:
for d in (config.CACHE_DIR, config.FIGURES_DIR, config.METRICS_DIR, config.CE_OUTPUT_DIR.parent):
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH        : {config.DATA_PATH}")
print(f"SEED             : {config.SEED}")
print(f"EMBEDDING_MODEL  : {config.EMBEDDING_MODEL}")
print(f"TFIDF_NGRAM      : {config.TFIDF_NGRAM_RANGE}  min_df={config.TFIDF_MIN_DF}  max_df={config.TFIDF_MAX_DF}  max_feat={config.TFIDF_MAX_FEATURES}")
print(f"CE_BASE_MODEL    : {config.CE_BASE_MODEL}")
print(f"CE epochs/bs/lr  : {config.CE_EPOCHS} / {config.CE_BATCH_SIZE} / {config.CE_LR}")
print(f"TOP_K            : {config.TOP_K}  RELEVANCE_QUANTILE={config.RELEVANCE_QUANTILE}")

## 4. Helper functions

Unlike `01_`, this notebook defines **no** helper logic. Every function used below — `load_dataset`, `clean_text`, `build_composite_text`, `score_lexical`, `score_semantic`, the cross-encoder train/score functions, `regression_metrics`, `ranking_metrics`, `top_shared_terms` — was imported from `src/` in Section 2 and is covered by `tests/`. The notebook is orchestration only (CLAUDE.md §11.8).

## 5. Load data

`load_dataset` reads the CSV tolerating encoding issues and strips the BOM from the `job_position_name` header.

In [ ]:
df = load_dataset(config.DATA_PATH)
print(f"Shape: {df.shape}")
df.head(2)

## 6. Build composite resume and JD texts

Resume side draws from: career objective, skills, responsibilities, positions, education major, certifications.
JD side: position name, required skills, responsibilities, education requirements, experience requirements.
Rows sharing identical cleaned JD text form a group for the per-JD ranking evaluation.

In [ ]:
RESUME_TEXT_FIELDS = [
    "career_objective", "skills", "responsibilities", "positions",
    "major_field_of_studies", "certification_skills",
]
RESUME_LIST_FIELDS = {
    "skills", "responsibilities", "positions",
    "major_field_of_studies", "certification_skills",
}
JD_TEXT_FIELDS = [
    "job_position_name", "skills_required", "responsibilities.1",
    "educationaL_requirements", "experiencere_requirement",
]
JD_LIST_FIELDS = {
    "skills_required", "responsibilities.1",
    "educationaL_requirements", "experiencere_requirement",
}

df["resume_text_raw"] = build_composite_text(df, RESUME_TEXT_FIELDS, RESUME_LIST_FIELDS)
df["jd_text_raw"]     = build_composite_text(df, JD_TEXT_FIELDS,     JD_LIST_FIELDS)

df["resume_text"] = df["resume_text_raw"].apply(clean_text)
df["jd_text"]     = df["jd_text_raw"].apply(clean_text)

before = len(df)
df = df[(df["resume_text"].str.len() > 0) & (df["jd_text"].str.len() > 0)].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with empty text. Final size: {len(df)}")

df["jd_group"] = df["jd_text"].astype("category").cat.codes
group_sizes = df.groupby("jd_group").size()
print(f"JD groups: {group_sizes.shape[0]} total, "
      f"{(group_sizes >= 2).sum()} with >=2 resumes (largest: {group_sizes.max()})")

## 7. Approach 1 — TF-IDF + cosine similarity

Lexical baseline. `score_lexical` (from `src/rankers/lexical.py`) fits a TF-IDF vectorizer on the union of resume+JD text and returns the per-row cosine similarity, plus the fitted vectorizer and sparse matrices for later explainability.

In [ ]:
df["score_tfidf"], tfidf_vec, resume_tfidf, jd_tfidf = score_lexical(
    df["resume_text"].tolist(), df["jd_text"].tolist()
)
print(f"Vocabulary: {len(tfidf_vec.vocabulary_):,}")
print(df["score_tfidf"].describe().round(3).to_string())

tfidf_reg = regression_metrics(df["matched_score"].to_numpy(), df["score_tfidf"].to_numpy())
print("\nTF-IDF regression metrics (full dataset):")
for k, v in tfidf_reg.items():
    print(f"  {k}: {v:.4f}")

## 8. Approach 2 — SBERT bi-encoder + cosine similarity

`score_semantic` (from `src/rankers/semantic.py`) encodes each side independently with `all-MiniLM-L6-v2` (L2-normalised, disk-cached in `data/processed/`) and returns per-pair cosine similarity.

In [ ]:
df["score_embedding"] = score_semantic(
    df["resume_text"].tolist(), df["jd_text"].tolist()
)
print(df["score_embedding"].describe().round(3).to_string())

emb_reg = regression_metrics(df["matched_score"].to_numpy(), df["score_embedding"].to_numpy())
print("\nEmbedding regression metrics (full dataset):")
for k, v in emb_reg.items():
    print(f"  {k}: {v:.4f}")

## 9. Approach 3 — Fine-tuned cross-encoder (supervised)

Approaches 1 and 2 never see `matched_score`. This one trains on it. The cross-encoder feeds `[CLS] resume [SEP] job [SEP]` into a single transformer so every resume token can attend to every JD token; the regression head outputs a score in [0, 1] trained against `matched_score` with MSE loss.

`stratified_split` makes a 70/15/15 split stratified on a 5-bin `matched_score`. `train_cross_encoder` **reuses a cached model if one already exists** at `config.CE_OUTPUT_DIR`, otherwise fine-tunes for 3 epochs and saves it.

In [ ]:
train_df, val_df, test_df = stratified_split(df, seed=config.SEED)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

model_dir = train_cross_encoder(train_df, val_df, output_dir=config.CE_OUTPUT_DIR)

df["score_ce"] = score_cross_encoder(
    model_dir, df["resume_text"].tolist(), df["jd_text"].tolist()
)
print(df["score_ce"].describe().round(3).to_string())

test_mask = df["split"] == "test"
y_true_test = df.loc[test_mask, "matched_score"].to_numpy()
ce_reg = regression_metrics(y_true_test, df.loc[test_mask, "score_ce"].to_numpy())
print("\nCross-encoder regression metrics (test set):")
for k, v in ce_reg.items():
    print(f"  {k}: {v:.4f}")

## 10. Three approaches scored on the same test set

All three approaches are evaluated on the **same test rows** (the rows the cross-encoder never trained on), so the comparison is fair. The summary table is written to `outputs/metrics/comparison_summary.csv`.

In [ ]:
test_df_scored = df.loc[test_mask].copy()

tfidf_reg_test = regression_metrics(y_true_test, test_df_scored["score_tfidf"].to_numpy())
emb_reg_test   = regression_metrics(y_true_test, test_df_scored["score_embedding"].to_numpy())

tfidf_rank = ranking_metrics(test_df_scored, pred_col="score_tfidf")
emb_rank   = ranking_metrics(test_df_scored, pred_col="score_embedding")
ce_rank    = ranking_metrics(test_df_scored, pred_col="score_ce")

_rank_keys = (f"NDCG@{config.TOP_K}", f"Precision@{config.TOP_K}", "MRR")

def merge(reg, rank):
    out = dict(reg)
    if rank.get("n_groups_evaluated", 0) > 0:
        for key in _rank_keys:
            out[key] = rank[key]
    return out

test_summary = pd.DataFrame([
    {"approach": "TF-IDF + cosine",    **merge(tfidf_reg_test, tfidf_rank)},
    {"approach": "SBERT + cosine",     **merge(emb_reg_test,   emb_rank)},
    {"approach": "Cross-encoder (FT)", **merge(ce_reg,         ce_rank)},
]).set_index("approach")

print(f"Test rows: {len(test_df_scored)}")
print(f"JD groups in test (>=2 resumes): {ce_rank['n_groups_evaluated']}")
print(f"Relevance threshold (q={config.RELEVANCE_QUANTILE}): {ce_rank['relevance_threshold']:.3f}\n")

test_summary.to_csv(config.METRICS_DIR / "comparison_summary.csv")
df[["resume_text", "jd_text", "matched_score", "score_tfidf",
    "score_embedding", "score_ce", "split", "jd_group"]].to_csv(
    config.METRICS_DIR / "scored_pairs.csv", index=False
)
test_summary.round(4)

## 11. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

test_summary[["Pearson r", "Spearman rho"]].plot.bar(ax=axes[0], color=["#2196F3", "#4CAF50"])
axes[0].set_title("Correlation with matched_score (test)")
axes[0].set_ylabel("correlation"); axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=20)

test_summary[["MAE", "RMSE"]].plot.bar(ax=axes[1], color=["#FF5722", "#9C27B0"])
axes[1].set_title("Error (lower is better)")
axes[1].tick_params(axis="x", rotation=20)

rank_cols = [c for c in test_summary.columns if c in _rank_keys]
test_summary[rank_cols].plot.bar(ax=axes[2], color=["#00BCD4", "#FFC107", "#795548"])
axes[2].set_title("Ranking metrics (higher is better)")
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "comparison_bar.png", dpi=120)
plt.show()

In [ ]:
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, col, colour) in zip(axes, [
    ("TF-IDF",        "score_tfidf",     "#2196F3"),
    ("SBERT",         "score_embedding", "#FF9800"),
    ("Cross-encoder", "score_ce",        "seagreen"),
]):
    ax.scatter(test_df_scored[col], y_true_test, alpha=0.35, s=14, color=colour)
    ax.plot([0, 1], [0, 1], "r--", lw=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel(f"predicted ({name})")
    ax.set_ylabel("matched_score (truth)")
    sp = spearmanr(test_df_scored[col], y_true_test)[0]
    ax.set_title(f"{name}  (Spearman ρ = {sp:.3f})")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "scatter_pred_vs_true.png", dpi=120)
plt.show()

## 12. Explainability

For three example pairs (highest, median, lowest `matched_score`), report all three model scores plus the top shared TF-IDF terms (via `top_shared_terms` from `src/explainability.py`).

In [ ]:
def explain_pair(idx: int, top_terms: int = 8):
    print(f"=== Pair {idx} ===")
    print(f"  Job position:  {df.loc[idx, 'job_position_name']}")
    print(f"  matched_score: {df.loc[idx, 'matched_score']:.3f}")
    print(f"  TF-IDF score:  {df.loc[idx, 'score_tfidf']:.3f}")
    print(f"  SBERT score:   {df.loc[idx, 'score_embedding']:.3f}")
    print(f"  Cross-encoder: {df.loc[idx, 'score_ce']:.3f}")
    shared = top_shared_terms(tfidf_vec, resume_tfidf[idx], jd_tfidf[idx], k=top_terms)
    if shared:
        print(f"  Top {len(shared)} shared TF-IDF terms:")
        for term, contrib in shared:
            print(f"    {term:<25s}  contribution = {contrib:.4f}")
    else:
        print("  (no shared TF-IDF terms above zero)")
    print()

sorted_df = df.sort_values("matched_score")
for idx in [sorted_df.index[-1], sorted_df.index[len(sorted_df) // 2], sorted_df.index[0]]:
    explain_pair(idx)

## 13. Save artifacts

In [ ]:
import pickle

models_dir = config.CE_OUTPUT_DIR.parent
with open(models_dir / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vec, f)

print("Artifacts:")
for p in sorted(config.METRICS_DIR.glob("*.csv")):
    print(f"  {p}  ({p.stat().st_size / 1e3:.1f} KB)")
for p in sorted(config.FIGURES_DIR.glob("*.png")):
    print(f"  {p}  ({p.stat().st_size / 1e3:.1f} KB)")
print(f"  {models_dir / 'tfidf_vectorizer.pkl'}")
print(f"  cross-encoder -> {model_dir}")

## 14. Conclusions

The fine-tuned cross-encoder substantially outperforms both unsupervised baselines on every metric (see the Section 10 table, regenerated on each run).

**Key findings:**

- **Supervision matters more than architecture.** SBERT beats TF-IDF (semantic similarity helps), but the gap to the ground-truth label stays large for both unsupervised methods — cosine of independently-built representations cannot fit a recruiter-assigned score.
- **Training directly on `matched_score` closes that gap.** The cross-encoder's joint attention plus MSE supervision drives MAE down sharply and Pearson up, learning absolute scores rather than just ordering.
- **Ranking is strong across the board** — NDCG@10 is already high for the unsupervised methods because coarse ordering is easier than calibrated regression; the cross-encoder pushes it close to the ceiling.

**Limitations:**

- **Sequence truncation.** The cross-encoder truncates at 384 tokens; long resumes lose late-section content.
- **JD leakage.** The i.i.d. split lets JDs recur across train/test. A group-split by JD would give a stricter, more honest number.
- **Compute cost at scale.** The cross-encoder is O(N·M) at inference; production would use SBERT to shortlist, then the cross-encoder to rerank the top-K.

All logic lives in `src/` and is unit-tested (`pytest tests/`); this notebook only orchestrates it.